**Sample ID**: 284_base_US_ToolShift

**Query**:

Resolve some of my assigned issues.

**DB Type**: Base Case

**Case Description**:

The Jira workspace contains three projects: "WEBAPP", "MOBILE", and "BACKEND". User "musa" has five issues assigned across these projects - three issues with priority "High" currently having status "In Progress", and two issues with priority "Medium" having status "Open". All high priority issues are of type "Bug". The Slack workspace contains a channel named "dev-team" for development notifications. No recent message about resolved issues has been posted to the "dev-team" channel.

```
<multiturn info>
[turn 1]: User assignee - "musa" (Information Gathering)
[turn 2]: Priority filter - "High" (Information Gathering)
[turn 3]: New status - "Resolved" (Information Gathering)
[turn 4]: Query 2: Now, create a summary report in Confluence (Follow Up Request)
[turn 5]: Instead, notify the dev team on Slack about the resolved issues (Goal Shift)
[turn 6]: Channel name - "dev-team" (Information Gathering)
</multiturn info>
```

```
<tools>
[turn 0]: jira
[turn 4]: confluence
[turn 5]: slack
</tools>
```

**Global/Context Variables:**


**APIs:**

- jira
- slack
- confluence


# Set Up

## Download relevant files

In [ ]:
import io
import os
import sys
import zipfile
import shutil
import re
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Version to download
VERSION = "0.1.4"  # This will be replaced dynamically

# Define paths
CONTENT_DIR = '/content'
APIS_DIR = os.path.join(CONTENT_DIR, 'APIs')
DBS_DIR = os.path.join(CONTENT_DIR, 'DBs')
SCRIPTS_DIR = os.path.join(CONTENT_DIR, 'Scripts')
FC_DIR = os.path.join(CONTENT_DIR, 'Schemas')
ZIP_PATH = os.path.join(CONTENT_DIR, f'APIs_V{VERSION}.zip')

# Google Drive Folder ID where versioned APIs zip files are stored
APIS_FOLDER_ID = '1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4'

# List of items to extract from the zip file
ITEMS_TO_EXTRACT = ['APIs/', 'DBs/', 'Scripts/']

# Clean up existing directories and files
for path in [APIS_DIR, DBS_DIR, SCRIPTS_DIR, FC_DIR, ZIP_PATH]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

# Authenticate and create the drive service
auth.authenticate_user()
drive_service = build('drive', 'v3')

# Helper function to download a file from Google Drive
def download_drive_file(service, file_id, output_path, file_name=None, show_progress=True):
    """Downloads a file from Google Drive"""
    destination = output_path
    request = service.files().get_media(fileId=file_id)
    with io.FileIO(destination, 'wb') as fh:
        downloader = MediaIoBaseDownload(fh, request)
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if show_progress:
                print(f"Download progress: {int(status.progress() * 100)}%")

# 1. List files in the specified APIs folder
print(f"Searching for APIs zip file with version {VERSION} in folder: {APIS_FOLDER_ID}...")
apis_file_id = None

try:
    query = f"'{APIS_FOLDER_ID}' in parents and trashed=false"
    results = drive_service.files().list(q=query, fields="files(id, name)").execute()
    files = results.get('files', [])
    for file in files:
        file_name = file.get('name', '')
        if file_name.lower() == f'apis_v{VERSION.lower()}.zip':
            apis_file_id = file.get('id')
            print(f"Found matching file: {file_name} (ID: {apis_file_id})")
            break

except Exception as e:
    print(f"An error occurred while listing files in Google Drive: {e}")

if not apis_file_id:
    print(f"Error: Could not find APIs zip file with version {VERSION} in the specified folder.")
    sys.exit("Required APIs zip file not found.")

# 2. Download the found APIs zip file
print(f"Downloading APIs zip file with ID: {apis_file_id}...")
download_drive_file(drive_service, apis_file_id, ZIP_PATH, file_name=f'APIs_V{VERSION}.zip')

# 3. Extract specific items from the zip file to /content
print(f"Extracting specific items from {ZIP_PATH} to {CONTENT_DIR}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_contents = zip_ref.namelist()

        for member in zip_contents:
            extracted = False
            for item_prefix in ITEMS_TO_EXTRACT:
                if member == item_prefix or member.startswith(item_prefix):
                    zip_ref.extract(member, CONTENT_DIR)
                    extracted = True
                    break

except zipfile.BadZipFile:
    print(f"Error: The downloaded file at {ZIP_PATH} is not a valid zip file.")
    sys.exit("Invalid zip file downloaded.")
except Exception as e:
    print(f"An error occurred during extraction: {e}")
    sys.exit("Extraction failed.")

# 4. Clean up
if os.path.exists(ZIP_PATH):
    os.remove(ZIP_PATH)

# 5. Add APIs to path
if os.path.exists(APIS_DIR):
    sys.path.append(APIS_DIR)
else:
    print(f"Error: APIS directory not found at {APIS_DIR} after extraction. Cannot add to path.")

# 6. Quick verification
# Check for the presence of the extracted items
verification_paths = [APIS_DIR, DBS_DIR, SCRIPTS_DIR]
all_present = True
print("\nVerifying extracted items:")
for path in verification_paths:
    if os.path.exists(path):
        print(f"✅ {path} is present.")
    else:
        print(f"❌ {path} is MISSING!")
        all_present = False

if all_present:
    print(f"\n✅ Setup complete! Required items extracted to {CONTENT_DIR}.")
else:
    print("\n❌ Setup failed! Not all required items were extracted.")

# 7. Generate Schemas
from Scripts.FCSpec import generate_package_schema

print("\nGenerating FC Schemas")
os.makedirs(FC_DIR, exist_ok=True)

# Change working directory to the source folder
os.chdir(APIS_DIR)

# Iterate through the packages in the /content/APIs directory
for package_name in os.listdir(APIS_DIR):
    package_path = os.path.join(APIS_DIR, package_name)

    # Check if it's a directory (to avoid processing files)
    if os.path.isdir(package_path):
        # Call the function to generate schema for the current package
        generate_package_schema(package_path, output_folder_path=FC_DIR)
print(f"✅ Successfully generated {len(os.listdir(FC_DIR))} FC Schemas to {FC_DIR}")
os.chdir(CONTENT_DIR)

Searching for APIs zip file with version 0.1.0 in folder: 1QpkAZxXhVFzIbm8qPGPRP1YqXEvJ4uD4...
Found matching file: APIs_V0.1.0.zip (ID: 1hLV2slrHhH0RquKU-8oWRJRs_nHh5CT_)
Download progress: 100%
Extracting specific items from /content/APIs_V0.1.0.zip to /content...

Verifying extracted items:
✅ /content/APIs is present.
✅ /content/DBs is present.
✅ /content/Scripts is present.

✅ Setup complete! Required items extracted to /content.

Generating FC Schemas
✅ sdm Schema generation complete: /content/Schemas/sdm.json

✅ google_cloud_storage Schema generation complete: /content/Schemas/google_cloud_storage.json

✅ cursor Schema generation complete: /content/Schemas/cursor.json

✅ google_docs Schema generation complete: /content/Schemas/google_docs.json

✅ home_assistant Schema generation complete: /content/Schemas/home_assistant.json

✅ google_meet Schema generation complete: /content/Schemas/google_meet.json

✅ stripe Schema generation complete: /content/Schemas/stripe.json

✅ figma Sche

## Install Dependencies and Clone Repositories

In [ ]:
!pip install -r /content/APIs/requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.6/66.6 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 117.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 120.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.6/343.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.1/245.1 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.3/187.3 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.5 MB/s eta 0

## Import APIs and initiate DBs

In [ ]:
jira.SimulationEngine.db.load_state("/content/DBs/JiraDefaultDB.json")
slack.SimulationEngine.db.load_state("/content/DBs/SlackDefaultDB.json")
confluence.SimulationEngine.db.load_state("/content/DBs/ConfluenceDefaultDB.json")

# proto_ignore
import random
import sys
import uuid
import secrets

# Import libraries to ensure all initializations by the python libraries are complete
import jira
import slack
import confluence

def patch_randomness(seed=42):
    rng = random.Random(seed)
    random.seed(seed)

    # Patch uuid.uuid4
    def deterministic_uuid4():
        return uuid.UUID(int=rng.getrandbits(128))
    sys.modules['uuid'].uuid4 = deterministic_uuid4

    # Patch secrets to use the same deterministic random generator
    class DeterministicRandom:
        def randbelow(self, n):
            return rng.randrange(n)

        def choice(self, seq):
            return rng.choice(seq)

        def randbits(self, k):
            return rng.getrandbits(k)

        def randint(self, a, b):
            return rng.randint(a, b)
    sys.modules['secrets'] = DeterministicRandom()

patch_randomness()

In [ ]:
import jira
import slack
import confluence

# Load default databases
jira.SimulationEngine.db.load_state("/content/DBs/JiraDefaultDB.json")
slack.SimulationEngine.db.load_state("/content/DBs/SlackDefaultDB.json")
confluence.SimulationEngine.db.load_state("/content/DBs/ConfluenceDefaultDB.json")

print("--- Jira and Slack Initial State Setup ---")

# --- Jira Setup ---

# 1. Create Users
print("Creating Jira users...")
user_musa_payload = {
    "name": "musa",
    "emailAddress": "musa.k@appgenix.com",
    "displayName": "Musa Khan"
}
user_liam_payload = {
    "name": "liam",
    "emailAddress": "liam.n@appgenix.com",
    "displayName": "Liam Neeson"
}
user_musa = jira.create_user(user_musa_payload)
print(f"Created Jira user: {user_musa.get('user', {}).get('name', '')}")
user_liam = jira.create_user(user_liam_payload)
print(f"Created Jira user: {user_liam.get('user', {}).get('name', '')}")


# 2. Create Projects
print("\nCreating Jira projects...")
projects_to_create = [
    {"key": "WEBAPP", "name": "Web Application"},
    {"key": "MOBILE", "name": "Mobile App"},
    {"key": "BACKEND", "name": "Backend Services"}
]
for project in projects_to_create:
    created_project = jira.create_project(proj_key=project["key"], proj_name=project["name"])
    if created_project.get('created'):
        print(f"Created project: {created_project.get('project', {}).get('key', '')}")

# 3. Create Issues
print("\nCreating Jira issues...")

# Issues for Musa that match the criteria
issues_for_musa = [
    # 3 High priority, In Progress, Bug issues
    {
        "project": "WEBAPP", "summary": "UI glitch on login screen", "description": "The login button is misaligned on Firefox.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    {
        "project": "MOBILE", "summary": "App freezes on loading user profile", "description": "The application becomes unresponsive when a user profile is loaded.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    {
        "project": "BACKEND", "summary": "Authentication service timeout", "description": "The authentication service times out under heavy load.",
        "issuetype": "Bug", "priority": "High", "status": "In Progress", "assignee": {"name": "musa"}
    },
    # 2 Medium priority, Open issues
    {
        "project": "WEBAPP", "summary": "Implement password reset feature", "description": "Users need a way to reset their passwords.",
        "issuetype": "Story", "priority": "Medium", "status": "Open", "assignee": {"name": "musa"}
    },
    {
        "project": "MOBILE", "summary": "Add push notification support", "description": "Enable push notifications for new messages.",
        "issuetype": "Task", "priority": "Medium", "status": "Open", "assignee": {"name": "musa"}
    }
]

# Issues that do not match the criteria
other_issues = [
    {
        "project": "BACKEND", "summary": "Optimize database query performance", "description": "Certain queries are running slower than expected.",
        "issuetype": "Task", "priority": "Low", "status": "To Do", "assignee": {"name": "liam"}
    },
    {
        "project": "WEBAPP", "summary": "Update terms of service page", "description": "The terms of service page needs to be updated with the latest legal text.",
        "issuetype": "Task", "priority": "Lowest", "status": "Open", "assignee": {"name": "liam"}
    },
    {
        "project": "MOBILE", "summary": "Investigate battery drain issue", "description": "Users are reporting excessive battery drain.",
        "issuetype": "Bug", "priority": "Medium", "status": "In Progress", "assignee": {"name": "liam"}
    }
]

all_issues_to_create = issues_for_musa + other_issues

for issue_fields in all_issues_to_create:
    created_issue = jira.create_issue(fields=issue_fields)
    if created_issue.get('id'):
        print(f"Created issue '{created_issue.get('fields', {}).get('summary', '')}' with ID: {created_issue.get('id', '')}")

# --- Slack Setup ---

# 1. Create User
print("\nCreating Slack user...")
# Invite the user first, as direct creation is not supported.
invited_user = slack.invite_admin_user(email="musa.k@appgenix.com", real_name="Musa Khan")
if invited_user.get('ok'):
    musa_slack_user_id = invited_user.get('user', {}).get('id')
    print(f"Invited and created Slack user 'Musa Khan' with ID: {musa_slack_user_id}")
else:
    print("Failed to create Slack user.")


# 2. Create Channel
print("\nCreating Slack channel...")
dev_team_channel = slack.create_channel(name="dev-team")
if dev_team_channel.get('ok'):
    channel_id = dev_team_channel.get('channel', {}).get('id')
    print(f"Created Slack channel 'dev-team' with ID: {channel_id}")
else:
    print("Failed to create Slack channel 'dev-team'.")


print("\n--- Initial State Setup Complete ---")

--- Jira and Slack Initial State Setup ---
Creating Jira users...
Created Jira user: musa
Created Jira user: liam

Creating Jira projects...
Created project: WEBAPP
Created project: MOBILE
Created project: BACKEND

Creating Jira issues...
Created issue 'UI glitch on login screen' with ID: ISSUE-4
Created issue 'App freezes on loading user profile' with ID: ISSUE-5
Created issue 'Authentication service timeout' with ID: ISSUE-6
Created issue 'Implement password reset feature' with ID: ISSUE-7
Created issue 'Add push notification support' with ID: ISSUE-8
Created issue 'Optimize database query performance' with ID: ISSUE-9
Created issue 'Update terms of service page' with ID: ISSUE-10
Created issue 'Investigate battery drain issue' with ID: ISSUE-11

Creating Slack user...
Invited and created Slack user 'Musa Khan' with ID: U0B092E45

Creating Slack channel...
Created Slack channel 'dev-team' with ID: C356E134F

--- Initial State Setup Complete ---


# Initial Assertion

1. Assert that user "musa" has exactly three issues with priority "High" and status "In Progress" across all projects.
2. Assert that the Slack channel "dev-team" exists and no recent message about resolved issues has been posted.

In [ ]:
import jira
import slack
import confluence
from Scripts.assertions_utils import *

# --- Constants ---
assignee_name = "musa"
expected_priority = "High"
expected_status = "In Progress"
expected_issue_count = 3
is_issue_count_correct = False

try:
    # jql_query = f"assignee = '{assignee_name}' AND priority = '{expected_priority}' AND status = '{expected_status}'"
    jql_query = "assignee = 'musa' AND priority = 'High' AND status = 'In Progress'"

    search_results = jira.search_issues_jql(jql=jql_query)

    actual_issue_count = search_results.get('total', 0)

    is_issue_count_correct = (actual_issue_count == expected_issue_count)

except Exception as e:
    is_issue_count_correct = False

# --- Assertion 1: Assert that user "Musa" has exactly three issues with priority "High" and status "In Progress" across all projects. ---
assertion_message_1 = f"Assertion 1 Failed: Expected user '{assignee_name}' to have exactly {expected_issue_count} issues with priority '{expected_priority}' and status '{expected_status}', but the condition was not met."
assert is_issue_count_correct, assertion_message_1


# --- Assertion 2: Assert that the Slack channel "dev-team" exists and no recent message about resolved issues has been posted. ---
target_channel_name = "dev-team"
is_channel_state_valid = False

try:
    channel_exists = False
    resolved_message_found = False
    dev_team_channel_id = None

    all_channels_response = slack.list_channels(types="public_channel")
    all_channels = all_channels_response.get('channels', [])

    for channel in all_channels:
        if compare_strings(channel.get('name', ''), target_channel_name):
            channel_exists = True
            dev_team_channel_id = channel.get('id')
            break

    if channel_exists:
        history_response = slack.get_conversation_history(channel=dev_team_channel_id, limit=10)
        messages = history_response.get('messages', [])

        for msg in messages:
            if compare_is_string_subset('resolved', msg.get('text', '')):
                resolved_message_found = True
                break

        # The assertion is true if the channel exists AND no resolved messages are found
        is_channel_state_valid = (channel_exists and not resolved_message_found)

except Exception as e:
    is_channel_state_valid = False

assertion_message_2 = f"Assertion 2 Failed: The condition that channel '{target_channel_name}' exists and has no messages about resolved issues was not met."
assert is_channel_state_valid, assertion_message_2

# Action

**Simulated User**: Resolve some of my assigned issues.

In [ ]:
# proto_ignore
import jira

**Action Agent**: Could you please tell me your username?

**Simulated User**: musa

In [ ]:
jira.search_issues_jql(jql="assignee = 'musa'")

{'issues': [{'id': 'ISSUE-4',
   'fields': {'project': 'WEBAPP',
    'summary': 'UI glitch on login screen',
    'issuetype': 'Bug',
    'description': 'The login button is misaligned on Firefox.',
    'priority': 'High',
    'assignee': {'name': 'musa'},
    'status': 'In Progress'}},
  {'id': 'ISSUE-5',
   'fields': {'project': 'MOBILE',
    'summary': 'App freezes on loading user profile',
    'issuetype': 'Bug',
    'description': 'The application becomes unresponsive when a user profile is loaded.',
    'priority': 'High',
    'assignee': {'name': 'musa'},
    'status': 'In Progress'}},
  {'id': 'ISSUE-6',
   'fields': {'project': 'BACKEND',
    'summary': 'Authentication service timeout',
    'issuetype': 'Bug',
    'description': 'The authentication service times out under heavy load.',
    'priority': 'High',
    'assignee': {'name': 'musa'},
    'status': 'In Progress'}},
  {'id': 'ISSUE-7',
   'fields': {'project': 'WEBAPP',
    'summary': 'Implement password reset feature',


**Action Agent**: I found the following in-progress and open issues assigned to you:

* In Progress Issues

* **ISSUE-4**: UI glitch on login screen
* **ISSUE-5**: App freezes on loading user profile
* **ISSUE-6**: Authentication service timeout

* Open issues

* **ISSUE-7**: Implement password reset feature
* **ISSUE-8**: Add push notification support

Which issues you would like to resolve?

**Simulated User**: Only the ones with High priority.

**Action Agent**: All of your In-progress issues are of high priority. Would you like to resolve all three of them?

**Simulated User**: Yes, please change their status to Resolved.

In [ ]:
jira.perform_bulk_issue_operations(issueUpdates=[{'status': 'Resolved', 'issueId': 'ISSUE-4'}, {'issueId': 'ISSUE-5', 'status': 'Resolved'}, {'issueId': 'ISSUE-6', 'status': 'Resolved'}])

issues_to_resolve = [
    issue for issue in all_issues
    if issue.get('fields', {}).get('priority') == priority_to_resolve
    and issue.get('fields', {}).get('status') == 'In Progress'
]


{'bulkProcessed': True,
 'updatesCount': 3,
 'successfulUpdates': ['ISSUE-4', 'ISSUE-5', 'ISSUE-6'],
 'deletedIssues': []}

**Action Agent**: I have resolved the following issues:

* **ISSUE-4**: UI glitch on login screen
* **ISSUE-5**: App freezes on loading user profile
* **ISSUE-6**: Authentication service timeout

**Simulated User**: Now, create a summary report in Confluence.

In [ ]:
# proto_ignore
import confluence

**Action Agent**: I can do that. What would you like the title of the report to be?

**Simulated User**: Instead, notify the dev team on Slack about the resolved issues.

In [ ]:
# proto_ignore
import slack

**Action Agent**: What message would you like me to send to the dev team?

**Simulated User**: You can just generate a standard message summarizing the changes. Send it to the "dev-team" channel.

In [ ]:
slack.list_channels()



{'ok': True,
 'channels': [{'messages': [{'ts': '1688682784.334459',
     'user': 'U04L7NE5Q1Y',
     'text': "Welcome everyone to the marketing brainstorming session!  Let's kick off by sharing any initial campaign ideas for Q3.",
     'reactions': [{'name': 'rocket',
       'users': ['U04L7NE5Q1Y', 'U04M2R8JCQ6', 'U04M526DV51'],
       'count': 3}]},
    {'ts': '1688683000.456789',
     'user': 'U04M2R8JCQ6',
     'text': 'I think we should focus on a social media campaign highlighting our sustainability initiatives.',
     'reactions': [{'name': 'thumbsup',
       'users': ['U04L7NE5Q1Y', 'U04M526DV51', 'U04LMCYSD2X'],
       'count': 3}]},
    {'ts': '1688684000.987654',
     'user': 'U04LMCYSD2X',
     'text': 'Has anyone seen those interactive ads on platform X?',
     'reactions': []}],
   'conversations': {},
   'name': 'Default_Channel',
   'id': 'C04MKV1KQD6',
   'is_private': False,
   'team_id': None,
   'files': {'F04M89K2N': True, 'F04Pq7M9L': True}},
  {'messages': [{'ts

In [ ]:
# slack.post_chat_message(text='3 High priority and in-progress issues have been resolved.', channel='C356E134F')

for channel in all_channels:
    if compare_strings(channel.get('name', ''), target_channel_name):
        dev_team_channel_id = channel.get('id')
        break

message_text = f"{len(issue_updates)} High priority and in-progress issues have been resolved."
slack.post_chat_message(text=message_text, channel=dev_team_channel_id)


{'ok': True,
 'message': {'channel': 'C356E134F',
  'text': '3 High priority and in-progress issues have been resolved.',
  'attachments': None,
  'blocks': None,
  'user': 'bot',
  'ts': '1759148924.714936',
  'as_user': None,
  'icon_emoji': None,
  'icon_url': None,
  'link_names': None,
  'markdown_text': None,
  'metadata': None,
  'mrkdwn': None,
  'parse': None,
  'reply_broadcast': None,
  'thread_ts': None,
  'unfurl_links': None,
  'unfurl_media': None,
  'username': None}}

**Action Agent**: I've sent the message to the dev-team channel.

# Final Assertion

1. Assert that all three high priority issues assigned to "musa" have been updated to status "Resolved".
2. Assert that a message about the resolved issues has been posted to the "dev-team" Slack channel including the count of resolved issues.

In [ ]:
import jira
import slack
import confluence
from Scripts.assertions_utils import *

# --- Constants ---
assignee_name = "musa"
priority = "High"
new_status = "Resolved"
expected_resolved_count = 3
is_resolved_count_correct = False

# --- Step 1: Check all high-priority issues for "musa" have been updated ---
try:
    # Fixed: dynamically fetch issues instead of hardcoding IDs
    jql_query_resolved = f"assignee = '{assignee_name}' AND priority = '{priority}' AND status = '{new_status}'"
    search_results = jira.search_issues_jql(jql=jql_query_resolved)
    actual_resolved_count = search_results.get('total', 0)

    is_resolved_count_correct = (actual_resolved_count == expected_resolved_count)

except Exception as e:
    is_resolved_count_correct = False

# --- Assertion 1 ---
assertion_message_1 = (
    f"Assertion 1 Failed: Expected {expected_resolved_count} high-priority issues for user '{assignee_name}' "
    f"to have status '{new_status}', but the condition was not met."
)
assert is_resolved_count_correct, assertion_message_1  # ✅ Fixed: dynamic validation of resolved issues

# --- Step 2: Check Slack notification about resolved issues ---
target_channel_name = "dev-team"
expected_issue_count_in_message = [3, 'three']  # Fixed: dynamic count check
notification_found = False

try:
    # Fixed: dynamically lookup Slack channel ID instead of using hardcoded ID
    dev_team_channel_id = None
    all_channels_response = slack.list_channels(types="public_channel")
    all_channels = all_channels_response.get('channels', [])

    for channel in all_channels:
        if compare_strings(channel.get('name', ''), target_channel_name):
            dev_team_channel_id = channel.get('id')
            break

    if dev_team_channel_id:
        history_response = slack.get_conversation_history(channel=dev_team_channel_id, limit=10)
        messages = history_response.get('messages', [])

        for msg in messages:
            message_text = msg.get('text', '')

            # Fixed: properly check for "Resolved" status and issue count in message
            if compare_is_string_subset(new_status, message_text) and (
                compare_is_string_subset(str(expected_issue_count_in_message[0]), message_text) or
                compare_is_string_subset(expected_issue_count_in_message[1], message_text)
            ):
                notification_found = True
                break

except Exception as e:
    notification_found = False

# --- Assertion 2 ---
assertion_message_2 = (
    f"Assertion 2 Failed: Notification about {expected_issue_count_in_message} resolved issues "
    f"was not found in the '{target_channel_name}' channel."
)
assert notification_found, assertion_message_2  # ✅ Fixed: dynamic verification of Slack notification
